In [0]:
# #Window functions — senior interview favorite
# %python

# from pyspark.sql import SparkSession
# from pyspark.sql.functions import *
# from pyspark.sql.types import *
# from pyspark.sql.window import Window

# # Running total + rank + lag/lead in one query
# w_customer = Window.partitionBy("customer_id").orderBy("order_date")
# w_rank = Window.partitionBy("status").orderBy(desc("amount"))

# result = spark.table("practice_db.orders").select(
#     "order_id", "customer_id", "amount", "order_date", "status",
#     sum("amount").over(w_customer).alias("running_total"),
#     rank().over(w_rank).alias("rank_in_status"),
#     lag("amount", 1).over(w_customer).alias("prev_order_amount"),
#     lead("amount", 1).over(w_customer).alias("next_order_amount"),
#     round(
#         (col("amount") - lag("amount",1).over(w_customer)) / lag("amount",1).over(w_customer) * 100, 2
#     ).alias("pct_change")
# )
# result.filter(col("customer_id") < 10).orderBy("customer_id","order_date").show(20)

In [0]:
# #Skew handling — critical for senior roles
# %python
# # Problem: skew_key=1 has ~10K rows, others have ~2 rows
# # Method 1: Salting
# salt_n = 10

# orders_salted = spark.table("practice_db.skewed_data") \
#     .withColumn("salt", (rand() * salt_n).cast("int")) \
#     .withColumn("salted_key", concat(col("skew_key").cast("string"), lit("_"), col("salt")))

# # Method 2: AQE (set in config — Spark 3.x)
# spark.conf.set("spark.sql.adaptive.enabled", "true")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
# spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")

# # Check partition distribution BEFORE vs AFTER
# spark.table("practice_db.skewed_data") \
#     .groupBy(spark_partition_id()).count() \
#     .orderBy(desc("count")).show(10)

In [0]:
# #Delta Lake — MERGE / time travel / Z-ordering

# %python
# from delta.tables import DeltaTable

# delta_orders = DeltaTable.forName(spark, "practice_db.orders")

# # MERGE (upsert pattern — very common interview question)
# updates_df = spark.range(1, 5001).select(
#     col("id").alias("order_id"),
#     lit("UPDATED").alias("status"),
#     (rand() * 5000).alias("amount")
# )

# delta_orders.alias("target").merge(
#     updates_df.alias("source"),
#     "target.order_id = source.order_id"
# ).whenMatchedUpdate(set={
#     "status": "source.status",
#     "amount": "source.amount"
# }).whenNotMatchedInsert(values={
#     "order_id": "source.order_id",
#     "status": "source.status",
#     "amount": "source.amount"
# }).execute()

# # Time travel
# spark.read.format("delta") \
#     .option("versionAsOf", 0) \
#     .table("practice_db.orders") \
#     .count()  # row count at version 0

# # Z-order for query optimization
# spark.sql("OPTIMIZE practice_db.orders ZORDER BY (customer_id, order_date)")

In [0]:
# #Scala — case class + tail recursion (asked at Hexaware/UST level)

# %scala
# import org.apache.spark.sql.{SparkSession, DataFrame}
# import org.apache.spark.sql.functions._

# // Case class schema definition
# case class Order(orderId: Long, customerId: Long, amount: Double, status: String)

# val spark = SparkSession.builder().getOrCreate()
# import spark.implicits._

# val ordersDS = spark.table("practice_db.orders").as[Order]

# // Typed transformation using Dataset API
# val highValueOrders = ordersDS
#   .filter(_.amount > 1000.0)
#   .map(o => o.copy(status = s"HIGH_VALUE_${o.status}"))

# // Tail recursive aggregation (interview classic)
# @annotation.tailrec
# def factorial(n: Long, acc: Long = 1L): Long =
#   if (n <= 1) acc else factorial(n - 1, n * acc)

# // Custom accumulator pattern
# val totalRevenue = ordersDS
#   .filter(_.status == "COMPLETED")
#   .map(_.amount)
#   .reduce(_ + _)

# println(s"Total revenue: $totalRevenue")



In [0]:
# %sql
# --SQL — complex analytical queries

# -- Cohort analysis: revenue by customer registration month
# WITH cohorts AS (
#   SELECT 
#     c.customer_id,
#     date_trunc('month', c.registered_date) AS cohort_month,
#     date_trunc('month', o.order_date)      AS order_month,
#     o.amount
#   FROM practice_db.customers c
#   JOIN practice_db.orders o ON c.customer_id = o.customer_id
#   WHERE o.status = 'COMPLETED'
# ),
# cohort_agg AS (
#   SELECT
#     cohort_month,
#     months_between(order_month, cohort_month) AS month_num,
#     count(DISTINCT customer_id)               AS active_customers,
#     round(sum(amount), 2)                     AS revenue
#   FROM cohorts
#   GROUP BY cohort_month, month_num
# )
# SELECT 
#   cohort_month,
#   month_num,
#   active_customers,
#   revenue,
#   round(revenue / first_value(revenue) OVER (PARTITION BY cohort_month ORDER BY month_num) * 100, 1) AS pct_of_first_month
# FROM cohort_agg
# ORDER BY cohort_month, month_num
# LIMIT 100;